# 01 - 用 PINN 求解一阶 ODE

目标方程：

$$\frac{du}{dt} = -u, \quad u(0)=1$$

解析解是：

$$u(t)=e^{-t}$$

PINN 的做法是让神经网络 `u_theta(t)` 同时满足：

- 方程残差：`du_theta/dt + u_theta = 0`
- 初值条件：`u_theta(0) = 1`

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim=1, out_dim=1, hidden_dim=32, depth=3):
        super().__init__()
        layers = [nn.Linear(in_dim, hidden_dim), nn.Tanh()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers += [nn.Linear(hidden_dim, out_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)

## 构造 PINN 损失

这里在区间 `[0, 5]` 内随机采样一批配点 collocation points，用它们计算方程残差。

In [ ]:
def ode_residual(model, t):
    t.requires_grad_(True)
    u = model(t)
    du_dt = torch.autograd.grad(
        u, t,
        grad_outputs=torch.ones_like(u),
        create_graph=True
    )[0]
    return du_dt + u

def exact_solution(t):
    return torch.exp(-t)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = []

for epoch in range(5000):
    t_f = 5.0 * torch.rand(128, 1, device=device)
    r = ode_residual(model, t_f)
    loss_phys = torch.mean(r**2)

    t0 = torch.zeros(1, 1, device=device)
    loss_ic = torch.mean((model(t0) - 1.0)**2)

    loss = loss_phys + 10.0 * loss_ic

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    history.append([loss.item(), loss_phys.item(), loss_ic.item()])
    if epoch % 1000 == 0:
        print(f"epoch={epoch:4d}, loss={loss.item():.3e}, phys={loss_phys.item():.3e}, ic={loss_ic.item():.3e}")

In [ ]:
t = torch.linspace(0, 5, 300, device=device).reshape(-1, 1)
with torch.no_grad():
    u_pred = model(t)
    u_exact = exact_solution(t)

history_t = torch.tensor(history)

plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(t.cpu(), u_exact.cpu(), label="exact")
plt.plot(t.cpu(), u_pred.cpu(), "--", label="PINN")
plt.xlabel("t")
plt.ylabel("u(t)")
plt.legend()

plt.subplot(1, 2, 2)
plt.semilogy(history_t[:, 0], label="total")
plt.semilogy(history_t[:, 1], label="physics")
plt.semilogy(history_t[:, 2], label="initial")
plt.legend()
plt.title("Loss terms")
plt.tight_layout()
plt.show()

max_error = torch.max(torch.abs(u_pred - u_exact)).item()
print(f"max absolute error = {max_error:.3e}")

## 练习

1. 把训练区间从 `[0, 5]` 改成 `[0, 10]`，观察误差是否变大。
2. 把初值损失权重 `10.0` 改成 `1.0` 或 `100.0`，观察初值是否满足得更好。
3. 把方程改成 `du/dt = -2u`，解析解应为 `exp(-2t)`。